In [18]:
df_usitc_clean = df_usitc_clean.iloc[1:].reset_index(drop=True)
df_usitc_clean[['2022','2023','2024','2025']] = df_usitc_clean[['2022','2023','2024','2025']].apply(pd.to_numeric)
print(df_usitc_clean[['hts_code', 'description', '2022', '2023', '2024', '2025']])

  hts_code                                        description        2022  \
0     8541  DIODES, TRANSISTORS AND SIMILAR DEVICES; PHOTO...   858896742   
1     8542  ELECTRONIC INTEGRATED CIRCUITS AND MICROASSEMB...  7968467106   
2     8543  ELECTRICAL MACHINES AND APPARATUS, HAVING INDI...   936776496   

         2023         2024         2025  
0  1121157665    678986464    665156957  
1  7186488555  11192001348  11994352150  
2   599175953    510508464    566799169  


In [19]:
df_dod = pd.read_csv('../data/dod_contracts_raw.csv', low_memory=False)
print(df_dod.shape)
print(df_dod[['naics_code', 'naics_description', 'award_id_piid', 'total_obligated_amount', 'recipient_name', 'product_or_service_code']].head(10))

(13133, 297)


KeyError: "['total_obligated_amount'] not in index"

In [21]:
print(df_dod[['naics_code', 'naics_description', 'award_id_piid', 'total_dollars_obligated', 'recipient_name', 'product_or_service_code']].head(10))

   naics_code                                  naics_description  \
0      334111                  ELECTRONIC COMPUTER MANUFACTURING   
1      334519  OTHER MEASURING AND CONTROLLING DEVICE MANUFAC...   
2      334511  SEARCH, DETECTION, NAVIGATION, GUIDANCE, AERON...   
3      334111                  ELECTRONIC COMPUTER MANUFACTURING   
4      334515  INSTRUMENT MANUFACTURING FOR MEASURING AND TES...   
5      334210                  TELEPHONE APPARATUS MANUFACTURING   
6      334111                  ELECTRONIC COMPUTER MANUFACTURING   
7      334111                  ELECTRONIC COMPUTER MANUFACTURING   
8      334511  SEARCH, DETECTION, NAVIGATION, GUIDANCE, AERON...   
9      334511  SEARCH, DETECTION, NAVIGATION, GUIDANCE, AERON...   

   award_id_piid  total_dollars_obligated  \
0  N0042124F1053                174808.67   
1  FA812224F0058               2928656.87   
2  FA252119FA290               3003828.44   
3  FA700024FG445                 29560.11   
4  FA851723D0005          

In [22]:
print(df_dod.shape)
print(df_dod['naics_code'].value_counts().head(10))

(13133, 297)
naics_code
334511    2705
334111    2500
334419    1333
334515     821
334290     800
334413     781
334516     753
334519     667
334220     510
334412     303
Name: count, dtype: int64


In [23]:
zsh: number expected


SyntaxError: invalid syntax (3905038636.py, line 1)

In [25]:
import pandas as pd

df = pd.read_csv('../data/census_semiconductor_imports.csv', skiprows=2)
df_usitc_clean = pd.read_excel('../data/usitc_semiconductor_imports.xlsx', sheet_name='CIF Import Value')
df_dod = pd.read_csv('../data/dod_contracts_raw.csv', low_memory=False)

print("Dataset 1 - Census imports:", df.shape)
print("Dataset 2 - USITC imports:", df_usitc_clean.shape)
print("Dataset 3 - DoD contracts:", df_dod.shape)

/Users/tylerbrown/Desktop/taiwan_blockade_project/venv/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Dataset 1 - Census imports: (12, 7)
Dataset 2 - USITC imports: (6, 8)
Dataset 3 - DoD contracts: (13133, 297)


In [26]:
import pandas as pd

# Load census data
df_census = pd.read_csv('../data/census_semiconductor_imports.csv', skiprows=2)

# Clean up column names
df_census.columns = ['commodity', 'country', 'year', 'customs_value', 'quantity', 'unit_value', 'drop']
df_census = df_census[['commodity', 'country', 'year', 'customs_value']]

# Remove commas and convert to numeric
df_census['customs_value'] = df_census['customs_value'].str.replace(',', '').astype(float)

print(df_census)

                                            commodity country  year  \
0   8541 Semicon Dev;photosens;led;mount Piezoelec...  Taiwan  2022   
1   8541 Semicon Dev;photosens;led;mount Piezoelec...  Taiwan  2023   
2   8541 Semicon Dev;photosens;led;mount Piezoelec...  Taiwan  2024   
3   8541 Semicon Dev;photosens;led;mount Piezoelec...  Taiwan  2025   
4   8542 Electronic Integrated Circuits & Microass...  Taiwan  2022   
5   8542 Electronic Integrated Circuits & Microass...  Taiwan  2023   
6   8542 Electronic Integrated Circuits & Microass...  Taiwan  2024   
7   8542 Electronic Integrated Circuits & Microass...  Taiwan  2025   
8   8543 Electrical Mach Etc, With Ind Functions N...  Taiwan  2022   
9   8543 Electrical Mach Etc, With Ind Functions N...  Taiwan  2023   
10  8543 Electrical Mach Etc, With Ind Functions N...  Taiwan  2024   
11  8543 Electrical Mach Etc, With Ind Functions N...  Taiwan  2025   

    customs_value  
0    8.460676e+08  
1    1.120813e+09  
2    6.817990e+0

In [27]:
# Extract HTS code as category label
df_census['hts_code'] = df_census['commodity'].str[:4]

# Add readable category names
category_names = {
    '8541': 'Discrete Semiconductors',
    '8542': 'Integrated Circuits',
    '8543': 'Specialized Electrical Machines'
}
df_census['chip_category'] = df_census['hts_code'].map(category_names)

# Filter to 2022-2024 only
df_census_24 = df_census[df_census['year'].isin([2022, 2023, 2024])]

# Average annual import value per category
df_avg = df_census_24.groupby(['hts_code', 'chip_category'])['customs_value'].mean().reset_index()
df_avg.columns = ['hts_code', 'chip_category', 'avg_annual_taiwan_imports']

print(df_avg)

  hts_code                    chip_category  avg_annual_taiwan_imports
0     8541          Discrete Semiconductors               8.828933e+08
1     8542              Integrated Circuits               8.883292e+09
2     8543  Specialized Electrical Machines               6.740674e+08


In [29]:
# Load and clean USITC data
df_usitc = pd.read_excel('../data/usitc_semiconductor_imports.xlsx', sheet_name='CIF Import Value')

# Extract just the data rows
df_usitc_clean = df_usitc.iloc[2:5].reset_index(drop=True)
df_usitc_clean.columns = ['metric', 'country', 'hts_code', 'description', '2022', '2023', '2024', '2025']
df_usitc_clean = df_usitc_clean[['hts_code', '2022', '2023', '2024']]
df_usitc_clean[['2022', '2023', '2024']] = df_usitc_clean[['2022', '2023', '2024']].apply(pd.to_numeric)

# Average annual Taiwan imports from USITC (for cross-reference)
df_usitc_clean['avg_annual_taiwan_imports_usitc'] = df_usitc_clean[['2022', '2023', '2024']].mean(axis=1)
df_usitc_clean['hts_code'] = df_usitc_clean['hts_code'].astype(str)

print(df_usitc_clean[['hts_code', 'avg_annual_taiwan_imports_usitc']])

  hts_code  avg_annual_taiwan_imports_usitc
0     8541                     8.863470e+08
1     8542                     8.782319e+09
2     8543                     6.821536e+08


/Users/tylerbrown/Desktop/taiwan_blockade_project/venv/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [31]:
df_total = pd.read_excel('../data/usitc_total_imports.xlsx', sheet_name='CIF Import Value')
print(df_total)

  Imports For Consumption|Annual Data  Unnamed: 1  \
0                      Data Row Count           3   
1                           Data Type  HTS Number   
2                    CIF Import Value        8541   
3                    CIF Import Value        8542   
4                    CIF Import Value        8543   
5                             Total:          NaN   

                                          Unnamed: 2      Unnamed: 3  \
0                                                NaN             NaN   
1                                        Description            2022   
2  DIODES, TRANSISTORS AND SIMILAR DEVICES; PHOTO...     17595544402   
3  ELECTRONIC INTEGRATED CIRCUITS AND MICROASSEMB...     42136860588   
4  ELECTRICAL MACHINES AND APPARATUS, HAVING INDI...     17168285255   
5                                                NaN  76,900,690,245   

       Unnamed: 4      Unnamed: 5  
0             NaN             NaN  
1            2023            2024  
2     266614384

/Users/tylerbrown/Desktop/taiwan_blockade_project/venv/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [32]:
# Clean total imports data
df_total_clean = df_total.iloc[2:5].reset_index(drop=True)
df_total_clean.columns = ['metric', 'hts_code', 'description', '2022', '2023', '2024']
df_total_clean[['2022', '2023', '2024']] = df_total_clean[['2022', '2023', '2024']].apply(pd.to_numeric)
df_total_clean['avg_annual_total_imports'] = df_total_clean[['2022', '2023', '2024']].mean(axis=1)
df_total_clean['hts_code'] = df_total_clean['hts_code'].astype(str)

# Merge with Taiwan imports
df_baseline = df_avg.merge(df_total_clean[['hts_code', 'avg_annual_total_imports']], on='hts_code')

# Calculate Taiwan share
df_baseline['taiwan_import_share_pct'] = (df_baseline['avg_annual_taiwan_imports'] / df_baseline['avg_annual_total_imports'] * 100).round(1)

print(df_baseline[['hts_code', 'chip_category', 'avg_annual_taiwan_imports', 'avg_annual_total_imports', 'taiwan_import_share_pct']])

  hts_code                    chip_category  avg_annual_taiwan_imports  \
0     8541          Discrete Semiconductors               8.828933e+08   
1     8542              Integrated Circuits               8.883292e+09   
2     8543  Specialized Electrical Machines               6.740674e+08   

   avg_annual_total_imports  taiwan_import_share_pct  
0              2.247546e+10                      3.9  
1              3.820097e+10                     23.3  
2              1.679811e+10                      4.0  


In [33]:
# Add TSMC platform context
df_baseline['tsmc_revenue_share_pct'] = [None, 43.0, None]  # HPC = 43% of TSMC revenue, maps to 8542
df_baseline['primary_us_end_use'] = ['Consumer electronics / IoT', 'Defense / HPC / Smartphones', 'Industrial / Consumer']

# Save to file
df_baseline.to_csv('../data/taiwan_baseline.csv', index=False)
print("Saved. Final baseline:")
print(df_baseline[['hts_code', 'chip_category', 'taiwan_import_share_pct', 'tsmc_revenue_share_pct', 'primary_us_end_use']])

Saved. Final baseline:
  hts_code                    chip_category  taiwan_import_share_pct  \
0     8541          Discrete Semiconductors                      3.9   
1     8542              Integrated Circuits                     23.3   
2     8543  Specialized Electrical Machines                      4.0   

   tsmc_revenue_share_pct           primary_us_end_use  
0                     NaN   Consumer electronics / IoT  
1                    43.0  Defense / HPC / Smartphones  
2                     NaN        Industrial / Consumer  
